# Titanic Survival Prediction
Example of training a random forest classifier using Apache Spark's MLlib library.


In [0]:
import pandas as pd
import mlflow
import mlflow.spark

from mlflow.models import infer_signature
from pyspark.sql.functions import col, median
from pyspark.ml.feature import StringIndexer, Imputer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

In [0]:
# Load the data into a Pandas DataFrame first, then convert it to a distributed Spark DataFrame

train_path = "/Workspace/Users/russellheines@gmail.com/databricks-practice/train.csv"

pdf = pd.read_csv(train_path)
df_spark = spark.createDataFrame(pdf)

# Split into 80% training and 20% testing data

train_data, test_data = df_spark.randomSplit([0.8, 0.2], seed=42)

In [0]:
# Set up preprocessing and model pipeline

sex_indexer = StringIndexer(inputCol="Sex", outputCol="SexIndex")
embarked_indexer = StringIndexer(inputCol="Embarked", outputCol="EmbarkedIndex", handleInvalid="keep")

age_imputer = Imputer(inputCols=["Age"], outputCols=["Age_Imputed"], strategy="median")
fare_imputer = Imputer(inputCols=["Fare"], outputCols=["Fare_Imputed"], strategy="median")

feature_cols = ["Pclass", "SexIndex", "Age_Imputed", "SibSp", "Parch", "Fare_Imputed", "EmbarkedIndex"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

rf = RandomForestClassifier(labelCol="Survived", featuresCol="features")

pipeline = Pipeline(stages=[sex_indexer, embarked_indexer, age_imputer, fare_imputer, assembler, rf])

In [0]:
# auto-logging is not supported on serverless
#mlflow.spark.autolog()

run_name = "titanic_sparkml_run"

with mlflow.start_run(run_name=run_name) as run:
    # Capture the active run information
    active_run_id = run.info.run_id
    print(f"Active Run ID: {active_run_id}")
    
    # Configure custom parameters inside our pipeline step
    rf.setNumTrees(100) 
    rf.setMaxDepth(5)
    
    # Train the model
    model = pipeline.fit(train_data)
    
    # Generate predictions for custom evaluations
    predictions = model.transform(test_data)

    # Calculate ROC-AUC
    binary_evaluator = BinaryClassificationEvaluator(
        labelCol="Survived",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    roc_auc = binary_evaluator.evaluate(predictions)
    mlflow.log_metric("test_roc_auc", roc_auc)

    print(f"SparkML Model ROC-AUC Accuracy: {roc_auc:.2%}")

    # Calculate Accuracy
    multiclass_evaluator = MulticlassClassificationEvaluator(
        labelCol="Survived",
        predictionCol="prediction",
        metricName="accuracy",
    )
    accuracy = multiclass_evaluator.evaluate(predictions)
    mlflow.log_metric("test_accuracy", accuracy)

    print(f"Test Accuracy: {accuracy:.4f}")

    # A signature is required by the UC model registry: raw feature columns in, predicted label out
    sig_in = test_data.select("Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked").na.drop().limit(5).toPandas()
    sig_out = predictions.select("prediction").limit(5).toPandas()

    mlflow.spark.log_model(
        model,
        "model",
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp",
        signature=infer_signature(sig_in, sig_out),
    )

In [0]:
# 1. Register the SparkML model
mlflow.register_model(
    model_uri=f"runs:/{active_run_id}/model",
    name="titanic_sparkml_model"
)